# S&P 500 Intelligence Oracle: Phase 3
## Data Processing & Feature Engineering

### Objectives
1. **Schema Alignment**: Convert market data date strings to Spark `DateType`.
2. **Technical Indicators**: Compute the **7-Day** and **30-Day Moving Averages** of the closing price to capture short- and medium-term trends.
3. **FNSPID Aggregation**: FNSPID contains multiple news articles per day across thousands of companies. We aggregate to a single daily record by computing:
   - `Daily_Sentiment_Score` — average numeric sentiment across all articles that day.
   - `Daily_Article_Count` — number of articles published (proxy for market attention).
4. **Data Integration**: Left-join the daily sentiment aggregates onto the market data by date.
5. **Final Preparation**: Fill nulls and persist the unified dataset for the ML model.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("SP500_Processing").getOrCreate()

market_df = spark.read.parquet("../data/processed/market_data_parquet")
sentiment_df = spark.read.parquet("../data/processed/sentiment_data_parquet")

print("Market data schema:")
market_df.printSchema()
print("FNSPID schema:")
sentiment_df.printSchema()

Market data schema:
root
 |-- Date: string (nullable = true)
 |-- Close: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Open: double (nullable = true)
 |-- Volume: long (nullable = true)

FNSPID schema:
root
 |-- Date: date (nullable = true)
 |-- Article_title: string (nullable = true)
 |-- Stock_symbol: string (nullable = true)
 |-- Publisher: string (nullable = true)
 |-- Sentiment_Score: double (nullable = true)



### Step 1 — Market Data: Schema Alignment & Technical Indicators

In [2]:
# Parse date string (SP500.csv uses M/d/yyyy format)
market_df = market_df.withColumn("Date", F.to_date(F.col("Date"), "M/d/yyyy"))

# Rolling windows ordered by date (single time-series partition)
w7  = Window.orderBy("Date").rowsBetween(-6,  0)
w30 = Window.orderBy("Date").rowsBetween(-29, 0)

market_df = (market_df
    .withColumn("Close_7D_Avg",  F.avg("Close").over(w7))
    .withColumn("Close_30D_Avg", F.avg("Close").over(w30))
)

print(f"Market rows: {market_df.count():,}")
market_df.select("Date", "Close", "Close_7D_Avg", "Close_30D_Avg").show(10)

+----------+-----------+------------------+------------------+
|      Date|      Close|      Close_7D_Avg|     Close_30D_Avg|
+----------+-----------+------------------+------------------+
|1927-12-30|17.65999985|       17.65999985|       17.65999985|
|1928-01-03|17.76000023|17.710000039999997|17.710000039999997|
|1928-01-04|17.71999931|       17.71333313|       17.71333313|
|1928-01-05|17.54999924|17.672499657499998|17.672499657499998|
|1928-01-06|17.65999985|17.669999695999998|17.669999695999998|
|1928-01-09|       17.5| 17.64166641333333| 17.64166641333333|
|1928-01-10|17.37000084|17.602857045714284|17.602857045714284|
|1928-01-11|17.35000038|17.558571407142857|17.571249962499998|
|1928-01-12|17.46999931| 17.51714270428571|17.559999889999997|
|1928-01-13|17.57999992| 17.49714279142857|      17.561999893|
+----------+-----------+------------------+------------------+
only showing top 10 rows


### Step 2 — FNSPID: Daily Aggregation

FNSPID already has a numeric `Sentiment_Score ∈ [−1, 1]` computed by VADER during ingestion. We simply group by date to get one row per calendar day.

In [3]:
# Aggregate to one row per date
daily_sentiment = (sentiment_df
    .groupBy("Date")
    .agg(
        F.avg("Sentiment_Score").alias("Daily_Sentiment_Score"),
        F.count("*").alias("Daily_Article_Count"),
    )
)

print(f"Unique dates with sentiment data: {daily_sentiment.count():,}")
daily_sentiment.orderBy("Date", ascending=False).show(10)

+----------+---------------------+-------------------+
|      Date|Daily_Sentiment_Score|Daily_Article_Count|
+----------+---------------------+-------------------+
|2020-06-11| -0.13880714285714285|                 14|
|2020-06-10|  0.03085673758865249|                282|
|2020-06-09|   0.2624577464788732|                284|
|2020-06-08|  0.25379151943462896|                283|
|2020-06-07|              0.04515|                  8|
|2020-06-06| -0.05253333333333333|                  3|
|2020-06-05|   0.5030920731707316|                328|
|2020-06-04|  0.21519836956521737|                184|
|2020-06-03|  0.40163865248226954|                282|
|2020-06-02|  0.09049045226130654|                199|
+----------+---------------------+-------------------+
only showing top 10 rows


### Step 3 — Join & Persist

In [4]:
# Left join: keep all market dates; sentiment is null for days with no FNSPID coverage
final_df = market_df.join(daily_sentiment, on="Date", how="left")

# Fill nulls: no news = neutral sentiment, zero article count
final_df = final_df.fillna({
    "Daily_Sentiment_Score": 0.0,
    "Daily_Article_Count":   0,
})

print("Integrated dataset schema:")
final_df.printSchema()

print("\nSample rows:")
final_df.select(
    "Date", "Close", "Close_7D_Avg", "Close_30D_Avg",
    "Daily_Sentiment_Score", "Daily_Article_Count"
).orderBy("Date", ascending=False).show(10)

print(f"\nTotal rows: {final_df.count():,}")
print(f"Rows with sentiment coverage (Article_Count > 0): "
      f"{final_df.filter(F.col('Daily_Article_Count') > 0).count():,}")

final_df.write.mode("overwrite").parquet("../data/processed/integrated_data_parquet")
print("\nData Integration Complete. Ready for Modeling.")


Data Integration Complete. Ready for Modeling.
